# Persona Vector Geometry

Analyses the geometric structure of PersVecGen steering vectors to answer:
1. **Manifold hypothesis** — do these vectors span a low-dimensional subspace?
2. **Similarity structure** — which personas are close / opposite in the residual stream?
3. **RL feasibility** — how many dimensions would we need to search over for coefficient optimisation?

No GPU required. Runs on CPU against pre-extracted `.pt` files.

In [ ]:
import os
import json
import pathlib

import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrowPatch
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize

# ── CONFIG ─────────────────────────────────────────────────────────────────────
# Point this at the directory containing *.pt and *.json trait files.
# Supports ${PERSONA_VECTORS_ROOT}/4bit  or  ${PERSONA_VECTORS_ROOT}/bf16
VECTORS_DIR = pathlib.Path(
    os.path.expandvars(os.environ.get('PERSONA_VECTORS_ROOT', '') + '/4bit')
).expanduser()

# Which layer to use for cross-trait comparison.
# 'auto' = per-trait best layer from companion .json; int = fixed layer for all traits.
LAYER = 'auto'  # or e.g. 29

print(f'VECTORS_DIR: {VECTORS_DIR}  (exists={VECTORS_DIR.exists()})')

## 1. Load vectors

In [ ]:
def best_layer_from_json(json_path: pathlib.Path) -> int | None:
    """Return layer with highest (max_coeff_score - baseline) from PersVecGen metadata."""
    if not json_path.exists():
        return None
    with open(json_path) as f:
        meta = json.load(f)
    sweep = meta.get('sweep_scores', {})
    if not sweep:
        return None
    def delta(lk):
        scores = {float(k): float(v) for k, v in sweep[lk].items()}
        baseline = scores.get(0.0, min(scores.values()))
        return max(scores.values()) - baseline
    return int(max(sweep.keys(), key=delta))


def load_trait_vectors(vectors_dir, layer='auto'):
    """
    Returns:
        traits      list of trait names
        vecs        np.ndarray  [n_traits, d_model]  (unit-normed)
        layers_used dict  {trait: layer_int}
    """
    pt_files = sorted(pathlib.Path(vectors_dir).glob('*.pt'))
    if not pt_files:
        raise FileNotFoundError(f'No .pt files found in {vectors_dir}')

    traits, raw_vecs, layers_used = [], [], {}
    for pt_path in pt_files:
        trait = pt_path.stem
        loaded = torch.load(str(pt_path), map_location='cpu', weights_only=False)

        if isinstance(loaded, dict):
            int_keys = {int(k): k for k in loaded}
            if layer == 'auto':
                bl = best_layer_from_json(pt_path.with_suffix('.json'))
                chosen = bl if (bl is not None and bl in int_keys) else max(int_keys)
            else:
                chosen = layer if layer in int_keys else max(int_keys)
            vec = loaded[int_keys[chosen]].float().numpy()
        else:
            chosen = -1
            vec = loaded.float().numpy()

        if vec.ndim > 1:
            vec = vec.squeeze()

        traits.append(trait)
        raw_vecs.append(vec)
        layers_used[trait] = chosen
        print(f'  {trait:30s}  layer={chosen:3d}  norm={np.linalg.norm(vec):.3f}  dim={vec.shape[0]}')

    V_raw = np.stack(raw_vecs)          # [n_traits, d_model]
    V     = normalize(V_raw, norm='l2') # unit-normalise for cosine geometry
    return traits, V, layers_used


traits, V, layers_used = load_trait_vectors(VECTORS_DIR, LAYER)
n_traits, d_model = V.shape
print(f'\nLoaded {n_traits} traits × {d_model}-dim vectors')
print(f'Layers used (unique): {sorted(set(layers_used.values()))}')

## 2. Cosine similarity heatmap

Which personas point in similar directions in the residual stream?

In [ ]:
import matplotlib.colors as mcolors

cos_sim = V @ V.T  # unit vectors → dot product = cosine similarity

# Cluster by hierarchical linkage so related traits sit together
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, leaves_list

dist = 1 - cos_sim
np.fill_diagonal(dist, 0)
dist = np.clip(dist, 0, None)  # numerical safety
Z = linkage(squareform(dist), method='average')
order = leaves_list(Z)

cos_ordered = cos_sim[np.ix_(order, order)]
labels_ordered = [traits[i] for i in order]

cmap = plt.cm.RdBu_r
fig, ax = plt.subplots(figsize=(max(8, n_traits * 0.6), max(7, n_traits * 0.6)))
im = ax.imshow(cos_ordered, cmap=cmap, vmin=-1, vmax=1)
ax.set_xticks(range(n_traits))
ax.set_yticks(range(n_traits))
ax.set_xticklabels(labels_ordered, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(labels_ordered, fontsize=9)
plt.colorbar(im, ax=ax, label='Cosine similarity')
ax.set_title(f'Persona vector cosine similarity ({d_model}-dim residual stream)', fontsize=11)

# annotate cells
for i in range(n_traits):
    for j in range(n_traits):
        v = cos_ordered[i, j]
        color = 'white' if abs(v) > 0.5 else 'black'
        ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=6.5, color=color)

plt.tight_layout()
plt.savefig('persona_cosine_similarity.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: persona_cosine_similarity.png')

# Most similar / most opposite pairs
sim_flat = [(cos_sim[i, j], traits[i], traits[j])
            for i in range(n_traits) for j in range(i+1, n_traits)]
sim_flat.sort(key=lambda x: -abs(x[0]))
print('\nTop 5 most similar pairs:')
for s, a, b in sim_flat[:5]:
    print(f'  {a:25s}  {b:25s}  cos={s:+.3f}')
print('\nTop 5 most opposite pairs:')
for s, a, b in sorted(sim_flat, key=lambda x: x[0])[:5]:
    print(f'  {a:25s}  {b:25s}  cos={s:+.3f}')

## 3. PCA — is there a low-dimensional manifold?

If the first few principal components capture most of the variance, the personas don't span the full residual-stream space — they live on a manifold. That manifold is the natural search space for RL/optimisation.

In [ ]:
# Full PCA over all n_traits dimensions
pca_full = PCA(n_components=min(n_traits, d_model))
pca_full.fit(V)

ev_ratio = pca_full.explained_variance_ratio_
ev_cumul = np.cumsum(ev_ratio)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Scree plot
ax = axes[0]
x = np.arange(1, len(ev_ratio) + 1)
ax.bar(x, ev_ratio * 100, color='#2a78d6', alpha=0.8, label='Component')
ax.plot(x, ev_cumul * 100, 'o-', color='#eb6834', lw=2, label='Cumulative')
ax.axhline(80, color='grey', ls='--', lw=0.8, label='80%')
ax.axhline(95, color='grey', ls=':', lw=0.8, label='95%')
ax.set_xlabel('Principal component')
ax.set_ylabel('Explained variance (%)')
ax.set_title('PCA scree plot — persona vectors')
ax.set_xticks(x)
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Cumulative only, with threshold annotations
ax = axes[1]
ax.plot(x, ev_cumul * 100, 'o-', color='#2a78d6', lw=2)
for thresh in (80, 90, 95):
    k = int(np.searchsorted(ev_cumul, thresh / 100)) + 1
    ax.axhline(thresh, color='grey', ls='--', lw=0.7, alpha=0.6)
    ax.text(x[-1] * 0.98, thresh + 0.5, f'{thresh}% @ PC{k}',
            ha='right', fontsize=8, color='#555')
ax.set_xlabel('Number of components')
ax.set_ylabel('Cumulative variance (%)')
ax.set_title('Cumulative explained variance')
ax.set_xticks(x)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('persona_pca_scree.png', dpi=150, bbox_inches='tight')
plt.show()

# Print summary
print('Component-by-component breakdown:')
for i, (ev, cum) in enumerate(zip(ev_ratio, ev_cumul), 1):
    bar = '█' * int(ev * 100 / max(ev_ratio) * 20)
    print(f'  PC{i:2d}  {ev*100:5.1f}%  cumul={cum*100:5.1f}%  {bar}')

for thresh in (80, 90, 95):
    k = int(np.searchsorted(ev_cumul, thresh / 100)) + 1
    print(f'\n→ {thresh}% variance explained by {k} components (of {n_traits} traits)')
    print(f'  Manifold dim ≈ {k}  |  RL search space shrinks from {d_model:,} to {k} dims')

## 4. 2-D PCA embedding — where do the personas sit?

In [ ]:
coords_2d = pca_full.transform(V)[:, :2]

fig, ax = plt.subplots(figsize=(9, 8))
ax.set_facecolor('#f8f8f5')
fig.patch.set_facecolor('#f8f8f5')

# Draw arrows from origin
for i, (x, y) in enumerate(coords_2d):
    ax.annotate('', xy=(x, y), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='#2a78d6', lw=1.4, alpha=0.55))

scatter = ax.scatter(coords_2d[:, 0], coords_2d[:, 1],
                     s=120, zorder=5, color='#2a78d6', alpha=0.85, edgecolors='white', lw=1)

# Label each point, offset to avoid overlap
from adjustText import adjust_text
texts = [ax.text(x, y, t, fontsize=9, ha='center')
         for t, (x, y) in zip(traits, coords_2d)]
try:
    adjust_text(texts, ax=ax, expand_points=(1.4, 1.4),
                arrowprops=dict(arrowstyle='-', color='#999', lw=0.5))
except Exception:
    pass  # adjustText optional

ax.axhline(0, color='#ccc', lw=0.7, ls='--')
ax.axvline(0, color='#ccc', lw=0.7, ls='--')
ax.set_xlabel(f'PC1  ({ev_ratio[0]*100:.1f}% var)', fontsize=10)
ax.set_ylabel(f'PC2  ({ev_ratio[1]*100:.1f}% var)', fontsize=10)
ax.set_title('Persona vectors — first two principal components', fontsize=12)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('persona_pca_2d.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: persona_pca_2d.png')

## 5. 3-D PCA embedding

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

coords_3d = pca_full.transform(V)[:, :3]

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

ax.scatter(*coords_3d.T, s=100, color='#2a78d6', alpha=0.85, edgecolors='white', lw=0.8)
for i, t in enumerate(traits):
    ax.text(*coords_3d[i], t, fontsize=7.5, alpha=0.85)
    ax.quiver(0, 0, 0, *coords_3d[i], length=1, normalize=False,
              color='#2a78d6', alpha=0.25, linewidth=0.8)

ax.set_xlabel(f'PC1 ({ev_ratio[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({ev_ratio[1]*100:.1f}%)')
ax.set_zlabel(f'PC3 ({ev_ratio[2]*100:.1f}%)')
ax.set_title('Persona vectors — first 3 PCs', fontsize=11)
plt.tight_layout()
plt.savefig('persona_pca_3d.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Intrinsic dimensionality estimates

More principled than picking a variance threshold.

In [ ]:
# ── Participation ratio (inverse participation ratio of eigenvalues) ────────────
# PR = (sum λ)² / sum λ²  — effective number of dimensions
eigenvalues = pca_full.explained_variance_
pr = eigenvalues.sum() ** 2 / (eigenvalues ** 2).sum()
print(f'Participation ratio (PR):  {pr:.2f}  (effective dims out of {n_traits})')

# ── Stable rank (nuclear norm² / Frobenius norm²) ─────────────────────────────
stable_rank = (np.sqrt(eigenvalues).sum()) ** 2 / eigenvalues.sum()
print(f'Stable rank:               {stable_rank:.2f}')

# ── Elbow detection (largest gap in cumulative variance) ──────────────────────
gaps = np.diff(ev_ratio)
elbow_k = int(np.argmin(gaps)) + 1  # component after which variance drops most steeply
print(f'Elbow at PC{elbow_k}: variance drops by {abs(gaps[elbow_k-1])*100:.2f}% after this')

# ── Reconstruction error at each truncation level ─────────────────────────────
print('\nReconstruction quality (mean cosine sim of reconstructed vs original):')
for k in range(1, n_traits + 1):
    coords_k = pca_full.transform(V)[:, :k]
    V_recon  = pca_full.mean_ + coords_k @ pca_full.components_[:k]
    V_recon_normed = normalize(V_recon, norm='l2')
    cos = (V_recon_normed * V).sum(axis=1).mean()
    bar = '█' * int(cos * 30)
    mark = ' ←' if k in (elbow_k, round(pr)) else ''
    print(f'  k={k:2d}  mean_cos={cos:.4f}  {bar}{mark}')

## 7. PCA component interpretation

What do the principal axes of persona space actually mean? Each PC is a direction in the residual stream — we can read off which traits load most positively/negatively.

In [ ]:
# Project each trait onto each PC and show the loadings
# loadings[i, j] = how much trait i contributes to PC j
loadings = pca_full.transform(V)   # [n_traits, n_components]

n_pcs_show = min(5, n_traits)
fig, axes = plt.subplots(1, n_pcs_show, figsize=(n_pcs_show * 3.5, max(5, n_traits * 0.4)))
if n_pcs_show == 1:
    axes = [axes]

for pc_idx, ax in enumerate(axes):
    scores = loadings[:, pc_idx]
    sort_idx = np.argsort(scores)
    colors = ['#e34948' if s < 0 else '#2a78d6' for s in scores[sort_idx]]
    ax.barh([traits[i] for i in sort_idx], scores[sort_idx], color=colors, alpha=0.85)
    ax.axvline(0, color='black', lw=0.8)
    ax.set_title(f'PC{pc_idx+1}\n({ev_ratio[pc_idx]*100:.1f}%)', fontsize=9)
    ax.tick_params(labelsize=7.5)
    ax.set_xlabel('Loading', fontsize=7)

plt.suptitle('Persona loadings on first principal components', fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig('persona_loadings.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: persona_loadings.png')

## 8. UMAP (optional — install `umap-learn`)

Non-linear embedding to catch any curved manifold structure PCA misses.

In [ ]:
try:
    import umap
    reducer = umap.UMAP(n_components=2, n_neighbors=min(5, n_traits - 1),
                        min_dist=0.1, metric='cosine', random_state=42)
    umap_coords = reducer.fit_transform(V)

    fig, ax = plt.subplots(figsize=(9, 7))
    ax.scatter(*umap_coords.T, s=120, color='#1baf7a', alpha=0.85,
               edgecolors='white', lw=1, zorder=5)
    texts = [ax.text(x, y, t, fontsize=9) for t, (x, y) in zip(traits, umap_coords)]
    try:
        from adjustText import adjust_text
        adjust_text(texts, ax=ax, expand_points=(1.4, 1.4),
                    arrowprops=dict(arrowstyle='-', color='#999', lw=0.5))
    except Exception:
        pass
    ax.set_title('Persona vectors — UMAP (cosine metric)', fontsize=11)
    ax.set_xlabel('UMAP-1'); ax.set_ylabel('UMAP-2')
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.savefig('persona_umap.png', dpi=150, bbox_inches='tight')
    plt.show()
except ImportError:
    print('umap-learn not installed — skipping UMAP.  Run: pip install umap-learn')

## 9. RL feasibility summary

Given the geometry, how tractable is coefficient optimisation for the Mafia game?

In [ ]:
# Manifold dimension at different variance thresholds
thresholds = [0.80, 0.90, 0.95]
dims = {t: int(np.searchsorted(ev_cumul, t)) + 1 for t in thresholds}

print('=' * 60)
print('RL FEASIBILITY SUMMARY')
print('=' * 60)
print(f'Number of traits (vectors): {n_traits}')
print(f'Residual stream dim:        {d_model:,}')
print(f'Participation ratio:        {pr:.1f}  (effective manifold dim)')
print()
for t, d in dims.items():
    episodes_cmaes = d * 10   # CMA-ES rule of thumb: ~10× dim evaluations
    episodes_bayes = d * 5    # Bayesian optimisation: more sample-efficient
    print(f'{int(t*100)}% variance threshold:')
    print(f'  Manifold dim = {d}')
    print(f'  CMA-ES needs ≈ {episodes_cmaes} function evaluations')
    print(f'  Bayesian Opt  ≈ {episodes_bayes} evaluations')
    print(f'  At 50 eps/eval → {episodes_cmaes // 50} CMA-ES runs  |  {episodes_bayes // 50} BO runs')
    print()

print('What you can optimise:')
print(f'  Level 1 — scalar coefficients for {n_traits} existing traits')
print(f'             dim={n_traits}, cheap, constrained to known directions')
print(f'  Level 2 — coordinates in {dims[0.90]}-PC manifold (90% var)')
print(f'             dim={dims[0.90]}, discovers interpolations between traits')
print(f'  Level 3 — full {d_model:,}-dim vector')
print(f'             NOT feasible with game-level reward')

## 10. Manifold coordinates for existing game configs

Project each current steering vector (used in mafia configs) into the PCA basis — these are the starting points for any optimisation run.

In [ ]:
n_manifold_dims = dims[0.90]   # choose 90% threshold
coords_manifold = pca_full.transform(V)[:, :n_manifold_dims]

print(f'Manifold coordinates ({n_manifold_dims}-D, 90% var explained):')
print(f'{"Trait":30s}', '  '.join([f'PC{i+1:4d}' for i in range(n_manifold_dims)]))
print('-' * (30 + 7 * n_manifold_dims))
for t, row in zip(traits, coords_manifold):
    vals = '  '.join([f'{v:+6.3f}' for v in row])
    print(f'{t:30s}  {vals}')

# Save as numpy for use in optimisation scripts
np.save('persona_manifold_coords.npy', coords_manifold)
np.save('persona_pca_components.npy', pca_full.components_[:n_manifold_dims])
np.save('persona_pca_mean.npy', pca_full.mean_)

import json
with open('persona_manifold_meta.json', 'w') as f:
    json.dump({'traits': traits, 'n_dims': n_manifold_dims,
               'explained_variance': ev_cumul[n_manifold_dims-1].item(),
               'layers_used': layers_used, 'd_model': d_model}, f, indent=2)

print('\nSaved:')
print('  persona_manifold_coords.npy   — [n_traits, n_dims] trait positions in PCA space')
print('  persona_pca_components.npy    — [n_dims, d_model] PCA basis vectors')
print('  persona_pca_mean.npy          — [d_model] mean vector')
print('  persona_manifold_meta.json    — metadata')
print()
print('To decode a manifold coordinate z back to a steering vector:')
print('  v = pca_mean + z @ pca_components  (then normalize if needed)')